In [1]:
import pandas as pd

In [5]:
analysis_df = pd.read_csv('./ecir_res/union_output_v2.csv')

In [6]:
gpp_df = analysis_df[analysis_df['Prediction Name']=='GPP']
rpp_df = analysis_df[analysis_df['Prediction Name']=='RPP']

In [102]:
from tools import fisher_test

def f(_x):
    _dict = {'0': 'QPP', '1': 'PerpC', '2': 'QualT5', '3': 'Readability'}
    output_list = [False, False, False, False]
    for i in _x[1:]:
        output_list[int(i)] = True
    return output_list

def significance(_n, _x, _b):
    _oa, _ob = fisher_test.compare_spearman_rhos(_n, _b, _x).values()
    return (_oa<0)&(_ob<0.1)

In [114]:
retr = 'e5'
k = 3

x = gpp_df.query('Use_Postgen==False').copy()
n = x['Number of Queries'].values[0]
x = x.reset_index(drop=True)
x = x[x['QA Task']=='nq']
x = x[x['Retriever']==retr]
x = x[x['Top-Retrieved Docs']==k]
identifier = f'{retr}%{k}'

best_single = x[x.Combination_Number=='p0123']['Best Single Rho'].values[0]
best_single_name = x[x.Combination_Number=='p0123']['Best Single Signal'].values[0]

x_a = pd.DataFrame(x.Combination_Number.apply(f).tolist(), columns=['QPP', 'PerpC', 'Qual', 'Read'])
x = pd.concat([x_a, x.reset_index(drop=True)], axis=1).drop(columns=['Prediction Name', 'QA Task', 'Top-Retrieved Docs', 'Number of Queries', \
                                                                 'Tau', 'Best Single Tau', 'Best Single Rho', 'Best Single Signal', \
                                                                 'Combination_Number', 'Use_Postgen', 'Retriever'])
x = x.rename(columns={'Rho': identifier})
best_row = pd.DataFrame([dict(zip(x.columns.tolist(), [False, False, False, False, best_single]))])
x = pd.concat([best_row, x], ignore_index=True)
x[f'Sig:{identifier}'] = x[identifier].apply(lambda _x: significance(n, _x, best_single))

In [115]:
from IPython.display import display, HTML
# Function to replace True/False with tick/blank
def tickmark(val):
    return "✔️" if val else ""

# Function to bold if Sig:e5%3 is True
def bold_sig(val, sig):
    return f"<b>{val:.6f}</b>" if sig else f"{val:.6f}"

# Apply formatting
df_display = x.copy()

# Replace first 4 columns with ticks
for col in ["QPP", "PerpC", "Qual", "Read"]:
    df_display[col] = df_display[col].apply(tickmark)

# Bold e5%3 where Sig:e5%3 is True
df_display[identifier] = [
    bold_sig(v, s) for v, s in zip(x[identifier], x[f"Sig:{identifier}"])
]

df_display = df_display.drop(columns=[f"Sig:{identifier}"])

# Display with HTML rendering
df_display = df_display.style.hide(axis="index").to_html()
display(HTML(df_display))

QPP,PerpC,Qual,Read,e5%3
,,,,0.228000
✔️,,,,0.279146
,✔️,,,0.152075
,,✔️,,0.128348
,,,✔️,0.084957
✔️,✔️,,,0.292300
✔️,,✔️,,0.285441
✔️,,,✔️,0.280996
,✔️,✔️,,0.159936
,✔️,,✔️,0.153703
